## Loading the Abalone Dataset

Physical shell measurements used to predict age (derived from ring count).

In [1]:
import pandas as pd
import numpy as np
headers = ["Sex","Length","Diameter","Height","Whole_weight","Shucked_weight","Viscera_weight","Shell_weight","Rings"]
abalone = pd.read_csv("abalone.data", header=None, names=headers)
print("Total samples:", len(abalone))
print("Feature names:", abalone.columns.tolist())
print(abalone.head().to_string())

Total samples: 4177
Feature names: ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings']
  Sex    Length  Diameter    Height  Whole_weight  Shucked_weight  Viscera_weight  Shell_weight  Rings
0   M  0.404191  0.616263  0.457104      1.314380        0.889172        0.574681      0.957132     22
1   I  0.436896  0.369024  0.329411      1.723120        0.353803        0.417678      0.752068     19
2   F  0.335221  0.376317  0.264878      1.193328        0.740234        0.204876      0.021514     17
3   M  0.748066  0.329315  0.601742      0.056013        0.326794        0.376329      0.404208     10
4   M  0.496027  0.248905  0.129101      2.785107        1.464169        0.529980      0.260170     25


## Age Derivation and Feature Selection

Age = Rings + 1.5. Three physical measurements selected as inputs.

In [1]:
abalone["age"] = abalone["Rings"] + 1.5
chosen_features = ["Length", "Diameter", "Whole_weight"]
X_raw = abalone[chosen_features].values
y_raw = abalone["age"].values.reshape(-1, 1)
print("Feature matrix shape:", X_raw.shape)
print("Target shape:", y_raw.shape)
print("Age range: %.1f – %.1f" % (y_raw.min(), y_raw.max()))

Feature matrix shape: (4177, 3)
Target shape: (4177, 1)
Age range: 2.5 – 29.5


## Train-Test Split and Normalization

80% training, 20% test. Normalization computed only from training data.

In [1]:
cutoff = int(0.8 * len(X_raw))
X_train, X_test = X_raw[:cutoff], X_raw[cutoff:]
y_train, y_test = y_raw[:cutoff], y_raw[cutoff:]
print("Training samples:", X_train.shape)
print("Test samples:    ", X_test.shape)
mu    = np.mean(X_train, axis=0)
sigma = np.std(X_train,  axis=0)
X_train = (X_train - mu) / sigma
X_test  = (X_test  - mu) / sigma
print("\nFeature means (train):", np.round(mu, 4))
print("Feature stds  (train):", np.round(sigma, 4))

Training samples: (3341, 3)
Test samples:     (836, 3)

Feature means (train): [0.439  0.3548 1.4425]
Feature stds  (train): [0.2127 0.1719 0.8106]


## Model and Loss

In [1]:
def linear_model(X, W, b):
    return X @ W + b

def mse(actual, predicted):
    return np.mean((actual - predicted) ** 2)

## Gradient Functions

In [1]:
def grad_weights(X, actual, predicted):
    diff = predicted - actual
    return (2.0 / len(actual)) * (X.T @ diff)

def grad_bias(actual, predicted):
    diff = predicted - actual
    return (2.0 / len(actual)) * np.sum(diff)

## Training Loop

In [1]:
W = np.random.randn(X_train.shape[1], 1) * 0.01
b = 0.0
lr = 0.1
n_epochs = 100
for ep in range(n_epochs):
    y_hat    = linear_model(X_train, W, b)
    cur_loss = mse(y_train, y_hat)
    dW       = grad_weights(X_train, y_train, y_hat)
    db       = grad_bias(y_train, y_hat)
    W        = W - lr * dW
    b        = b - lr * db
    if ep % 10 == 0:
        print(f"Epoch {ep:3d}  Loss: {cur_loss:.4f}")

Epoch   0  Loss: 322.4569
Epoch  10  Loss: 68.3036
Epoch  20  Loss: 65.3735
Epoch  30  Loss: 65.3397
Epoch  40  Loss: 65.3393
Epoch  50  Loss: 65.3393
Epoch  60  Loss: 65.3393
Epoch  70  Loss: 65.3393
Epoch  80  Loss: 65.3393
Epoch  90  Loss: 65.3393


## Test Set Evaluation

In [1]:
test_preds = linear_model(X_test, W, b)
test_mse = mse(y_test, test_preds)
test_mae = np.mean(np.abs(y_test - test_preds))
print(f"Test MSE : {test_mse:.4f}")
print(f"Test MAE : {test_mae:.4f}")
print("\nSample | True Age | Pred Age | Error")
for i in range(5):
    t = y_test[i][0]
    p = test_preds[i][0]
    print(f"  {i+1}    |  {t:6.2f}  |  {p:6.2f}  | {abs(t-p):.2f}")

Test MSE : 67.9475
Test MAE : 7.1606

Sample | True Age | Pred Age | Error
  1    |    6.50  |   15.86  | 9.36
  2    |   27.50  |   16.02  | 11.48
  3    |   12.50  |   16.12  | 3.62
  4    |   17.50  |   16.05  | 1.45
  5    |   21.50  |   16.23  | 5.27
